# Continuous EV Copilot chat
Run the cells from top to bottom. The final cell starts an interactive chat. Type `quit` to end it and stop SUMO. Each question uses the same LangGraph thread; tool calls, tool outputs, and the final answer are streamed.

In [1]:
from pathlib import Path
import json
import sys
from uuid import uuid4

from dotenv import load_dotenv
from IPython.display import JSON, Markdown, display
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "agent" / "graph.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the vehicle_ai_copilote project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")
print("Project root:", PROJECT_ROOT)

Project root: c:\Users\Yessine\Desktop\vehicle_ai_copilote


## Start the simulated EV
Set `USE_GUI = True` to watch SUMO. This starts a fresh simulated trip; it does not advance the vehicle automatically between questions.

In [2]:
import traci
from simulation.trip_simulator import TripSimulator

if traci.isLoaded():
    traci.close(False)

USE_GUI = False
simulator = TripSimulator(
    origin="Route El Ain, central Sfax",
    destination="North-east central Sfax",
    traffic_level="normal",
    initial_soc_percent=80.0,
    ambient_temperature_c=25.0,
)
initial_state = simulator.start(use_gui=USE_GUI)
print("SUMO status:", initial_state["status"])
print("Battery SOC:", initial_state["ev"]["battery_soc_percent"], "%")

SUMO status: running
Battery SOC: 80.0 %


## Build the copilot and stream both graphs
The notebook traces each internal energy node when the agent calls energy_consumption_kwh. Set TRACE_ALL_SEGMENTS = True to print every segment. The LLM still receives the compact energy result.


In [3]:
from agent.graph import build_copilot_graph
from simulation.energy_estimation import build_energy_estimation_graph
import agent.tools as agent_tools

TRACE_ALL_SEGMENTS = False
energy_graph = build_energy_estimation_graph(simulator)

class TracedEnergyGraph:
    """Stream internal updates from the exact calculation used by the tool."""

    def __init__(self, graph):
        self.graph = graph
        self.last_state = None

    def invoke(self, input_state):
        final_state = dict(input_state)
        for update in self.graph.stream(input_state, stream_mode="updates"):
            for node_name, changes in update.items():
                print(f"  [Energy subgraph] {node_name}")
                if not isinstance(changes, dict):
                    continue
                final_state.update(changes)
                if changes.get("errors"):
                    print("    errors:", changes["errors"])
                    continue
                if node_name == "fetch_fresh_context":
                    route = changes.get("route", {})
                    print("    resolved origin:", changes.get("origin"))
                    print("    resolved destination:", changes.get("destination"))
                    print("    route km:", route.get("distance_km"))
                    print("    weather ?C:", changes.get("weather", {}).get("temperature_c"))
                    vehicle = changes.get("vehicle", {})
                    print("    battery kWh:", vehicle.get("ev", {}).get("battery_energy_kwh"))
                    print("    traffic:", vehicle.get("traffic", {}).get("level"))
                elif node_name in {"segment_route", "apply_speed_limits", "prepare_features", "predict_segments"}:
                    segments = changes.get("route_segments", [])
                    print("    segments:", len(segments))
                    sample = segments if TRACE_ALL_SEGMENTS else segments[:3]
                    for segment in sample:
                        row = {
                            "id": segment.get("segment_id"),
                            "road_type": segment.get("road_type"),
                            "distance_km": segment.get("distance_km"),
                            "route_speed_kmh": segment.get("route_speed_kmh"),
                            "estimated_speed_kmh": segment.get("estimated_speed_kmh"),
                        }
                        if "ml_features" in segment:
                            row["ml_features"] = segment["ml_features"]
                        if "predicted_energy_kwh" in segment:
                            row["predicted_energy_kwh"] = segment["predicted_energy_kwh"]
                        print("   ", row)
                    if len(segments) > len(sample):
                        print(f"    ... {len(segments) - len(sample)} more; set TRACE_ALL_SEGMENTS = True for all")
                elif node_name == "aggregate_energy":
                    print("    total predicted kWh:", changes.get("total_predicted_energy_kwh"))
                elif node_name == "check_feasibility":
                    for key in ("available_energy_kwh", "reserve_energy_kwh", "energy_after_trip_kwh", "projected_arrival_soc_percent", "feasible"):
                        print(f"    {key}:", changes.get(key))
        self.last_state = final_state
        return final_state

traced_energy_graph = TracedEnergyGraph(energy_graph)
original_energy_builder = agent_tools.build_energy_estimation_graph
try:
    agent_tools.build_energy_estimation_graph = lambda _sim: traced_energy_graph
    copilot = build_copilot_graph(simulator)
finally:
    agent_tools.build_energy_estimation_graph = original_energy_builder

THREAD_ID = f"continuous-chat-{uuid4()}"
print("Thread:", THREAD_ID)

def ask_copilot(question: str):
    display(Markdown(f"**Driver:** {question}"))
    final_message = None
    config = {"configurable": {"thread_id": THREAD_ID}}
    for update in copilot.stream(
        {"messages": [HumanMessage(content=question)]},
        config=config,
        stream_mode="updates",
    ):
        for node_name, payload in update.items():
            for message in payload.get("messages", []):
                if isinstance(message, AIMessage) and message.tool_calls:
                    for call in message.tool_calls:
                        print(f"[Agent ? tool] {call['name']} args={call.get('args', {})}")
                elif isinstance(message, ToolMessage):
                    print(f"[Tool ? agent] {message.name}")
                    try:
                        result = json.loads(message.content) if isinstance(message.content, str) else message.content
                        display(JSON(result, expanded=False))
                    except (TypeError, ValueError):
                        print(message.content)
                elif node_name == "assistant" and isinstance(message, AIMessage) and message.content:
                    final_message = message
    if final_message is not None:
        display(Markdown("**Copilot:**\n\n" + str(final_message.content)))
    return final_message

Thread: continuous-chat-141a94a3-fc68-40f2-a443-92b43efd6dd8


## Continuous chat
Type a question, `/step 30` to advance SUMO by 30 simulated seconds, or `quit` to end the chat. The simulation is stopped when this cell ends.

In [4]:
def start_chat() -> None:
    print("EV Copilot ready. Type quit to stop, or /step <seconds> to move the car.")
    while True:
        try:
            question = input("\nDriver: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nChat ended.")
            break
        if not question:
            continue
        if question.lower() in {"quit", "exit", "stop"}:
            print("Chat ended.")
            break
        if question.lower().startswith("/step"):
            parts = question.split()
            if len(parts) != 2:
                print("Usage: /step <seconds>")
                continue
            try:
                seconds = float(parts[1])
                if seconds <= 0:
                    raise ValueError
                simulator.step(seconds)
                print(f"Advanced SUMO by {seconds:g} simulated seconds.")
            except ValueError:
                print("Enter a positive number of seconds.")
            continue
        try:
            ask_copilot(question)
        except Exception as error:
            print(f"{type(error).__name__}: {error}")

try:
    start_chat()
finally:
    simulator.stop()
    print("SUMO stopped.")

EV Copilot ready. Type quit to stop, or /step <seconds> to move the car.


**Driver:** energy required from sfax to sousse

[Agent ? tool] energy_consumption_kwh args={'destination': 'sousse', 'origin': 'sfax'}
  [Energy subgraph] fetch_fresh_context
    resolved origin: صفاقس, المدينة, معتمدية صفاقس المدينة, ولاية صفاقس, 3000, تونس
    resolved destination: سوسة, معتمدية سوسة المدينة, ولاية سوسة, تونس
    route km: 134.05
    weather ?C: 21.4
    battery kWh: 33.76
    traffic: normal
  [Energy subgraph] segment_route
    segments: 34
    {'id': 1, 'road_type': 'urban', 'distance_km': 0.087, 'route_speed_kmh': 10.0, 'estimated_speed_kmh': None}
    {'id': 2, 'road_type': 'urban', 'distance_km': 0.027, 'route_speed_kmh': 30.0, 'estimated_speed_kmh': None}
    {'id': 3, 'road_type': 'urban', 'distance_km': 0.263, 'route_speed_kmh': 22.2, 'estimated_speed_kmh': None}
    ... 31 more; set TRACE_ALL_SEGMENTS = True for all
  [Energy subgraph] apply_speed_limits
    segments: 34
    {'id': 1, 'road_type': 'urban', 'distance_km': 0.087, 'route_speed_kmh': 10.0, 'estimated_speed_kmh': 10.0}
    {'id': 2, 'road_typ

<IPython.core.display.JSON object>

**Copilot:**

**Estimated energy required:** ≈ **28.5 kWh** for the 134 km drive from Sfax to Sousse.

**Why this matters**

- Your vehicle currently has **33.8 kWh** of usable energy.
- A 15 % reserve (≈ 6.3 kWh) is kept, leaving **27.4 kWh** available for the trip.
- The trip would need **28.5 kWh**, which exceeds the post‑reserve energy by about **1 kWh**.

**Result:** The journey is **not feasible** with the present state of charge; you would arrive with only about **12 % SOC** (≈ 5.3 kWh left), below the reserve threshold.

**What to do**

- Charge the battery before departing (add at least ~1 kWh, preferably more to keep a safety margin).
- Or consider a shorter leg, a charging stop en‑route, or a lower‑speed driving profile to reduce consumption.

Chat ended.
SUMO stopped.


## Compare subgraph and direct model predictions
Run this after ending chat with quit. It reuses the exact 15 feature values and segment distances from the last successful energy-tool call. It loads the same saved model separately; no routing, weather, SUMO, or LLM call is repeated.


In [5]:
from math import isclose
import pandas as pd
from simulation.energy_estimation import FEATURES, SavedEnergyModel

state = traced_energy_graph.last_state
if not state or state.get("errors") or not state.get("route_segments"):
    print("No completed energy estimate captured. Ask an energy question in chat first.")
else:
    segments = state["route_segments"]
    direct_model = SavedEnergyModel()
    feature_rows = []
    comparison_rows = []
    for segment in segments:
        features = segment["ml_features"]
        feature_rows.append({"segment_id": segment["segment_id"], **{name: features[name] for name in FEATURES}})
        direct_rate = direct_model.predict(features)  # kWh/km from the same saved model
        distance = float(segment["distance_km"])
        direct_energy = direct_rate * distance
        subgraph_energy = float(segment["predicted_energy_kwh"])
        comparison_rows.append({
            "segment_id": segment["segment_id"],
            "distance_km": distance,
            "direct_rate_kwh_per_km": direct_rate,
            "direct_energy_kwh": direct_energy,
            "subgraph_energy_kwh": subgraph_energy,
            "difference_kwh": direct_energy - subgraph_energy,
            "matches": isclose(direct_energy, subgraph_energy, rel_tol=1e-9, abs_tol=1e-9),
        })

    features_df = pd.DataFrame(feature_rows, columns=["segment_id", *FEATURES])
    comparison_df = pd.DataFrame(comparison_rows)
    display(Markdown("### Exact ML inputs captured from the subgraph"))
    display(features_df)
    display(Markdown("### Direct prediction versus subgraph prediction"))
    display(comparison_df)
    direct_total = comparison_df["direct_energy_kwh"].sum()
    subgraph_total = float(state["total_predicted_energy_kwh"])
    print("Segments:", len(comparison_df))
    print(f"Direct total: {direct_total:.6f} kWh")
    print(f"Subgraph total: {subgraph_total:.6f} kWh")
    print(f"Difference: {direct_total - subgraph_total:.12f} kWh")
    print("All segment predictions match:", bool(comparison_df["matches"].all()))
    print("Trip totals match:", isclose(direct_total, subgraph_total, rel_tol=1e-9, abs_tol=1e-9))

### Exact ML inputs captured from the subgraph

,segment_id,motor_power_kW,vehicle_weight_kg,drag_coefficient_Cd,frontal_area_m2,rolling_resistance_Cr,motor_efficiency_pct,avg_driving_speed_kmh,ambient_temperature_C,ac_usage,terrain_type,traffic_condition,driving_style,passengers,battery_age_years,tyre_pressure_deviation_psi
0,1,125.0,1270.0,0.29,2.38,0.012,92.0,10.0,21.4,0,0,1,1,1.0,1.0,0.0
1,2,125.0,1270.0,0.29,2.38,0.012,92.0,30.0,21.4,0,0,1,1,1.0,1.0,0.0
2,3,125.0,1270.0,0.29,2.38,0.012,92.0,22.2,21.4,0,0,1,1,1.0,1.0,0.0
3,4,125.0,1270.0,0.29,2.38,0.012,92.0,28.1,21.4,0,0,1,1,1.0,1.0,0.0
4,5,125.0,1270.0,0.29,2.38,0.012,92.0,30.0,21.4,0,0,1,1,1.0,1.0,0.0
5,6,125.0,1270.0,0.29,2.38,0.012,92.0,14.0,21.4,0,0,1,1,1.0,1.0,0.0
6,7,125.0,1270.0,0.29,2.38,0.012,92.0,30.0,21.4,0,0,1,1,1.0,1.0,0.0
7,8,125.0,1270.0,0.29,2.38,0.012,92.0,14.0,21.4,0,0,1,1,1.0,1.0,0.0
8,9,125.0,1270.0,0.29,2.38,0.012,92.0,30.1,21.4,0,0,1,1,1.0,1.0,0.0
9,10,125.0,1270.0,0.29,2.38,0.012,92.0,22.0,21.4,0,0,1,1,1.0,1.0,0.0


### Direct prediction versus subgraph prediction

,segment_id,distance_km,direct_rate_kwh_per_km,direct_energy_kwh,subgraph_energy_kwh,difference_kwh,matches
0,1,0.087,0.116008,0.010093,0.010093,-1.734723e-18,True
1,2,0.027,0.116225,0.003138,0.003138,0.000000e+00,True
2,3,0.263,0.115587,0.030399,0.030399,0.000000e+00,True
3,4,0.414,0.116164,0.048092,0.048092,6.938894e-18,True
4,5,0.196,0.116225,0.022780,0.022780,-3.469447e-18,True
5,6,0.009,0.116008,0.001044,0.001044,-4.336809e-19,True
6,7,2.891,0.116225,0.336007,0.336007,0.000000e+00,True
7,8,0.027,0.116008,0.003132,0.003132,0.000000e+00,True
8,9,5.850,0.116114,0.679264,0.679264,-1.110223e-16,True
9,10,0.024,0.115617,0.002775,0.002775,-4.336809e-19,True


Segments: 34
Direct total: 28.475084 kWh
Subgraph total: 28.475084 kWh
Difference: 0.000000000000 kWh
All segment predictions match: True
Trip totals match: True
